# Calibration & Position Debiasing

1. Reliability diagrams before/after Platt scaling
2. ECE improvement (target: < 0.03 after scaling)
3. Position invariance: same (user, restaurant) at positions 1-10

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, Markdown

import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import set_seed, DEFAULT_HPARAMS, MODELS_DIR
from src.models.deepfm import DeepFM
from src.models.calibration import PlattScaler, PositionDebiaser, compute_ece
from src.training.trainer import Trainer, TrainerConfig, AdClickDataset, _collate_batch
from src.training.run_training import infer_feature_config
from torch.utils.data import DataLoader

SEED = 42
set_seed(SEED)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/anam/Desktop/Dev/yelp-ads-bidding-ctr/yelp-ads-bidding-ctr/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/anam/Desktop/Dev/yelp-ads-bidding-ctr/yelp-ads-bidding-ctr/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/anam/Desktop/Dev/yelp-ads-bidding-c

In [2]:
data_path = PROJECT_ROOT / "data" / "processed" / "ad_impressions.parquet"
raw_df = pd.read_parquet(data_path)
df = raw_df.copy()
feature_config = infer_feature_config(df, embedding_dim=DEFAULT_HPARAMS.embedding_dim)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print("train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

train: (350000, 21) val: (75000, 21) test: (75000, 21)


In [3]:
# Train a fresh DeepFM (or load checkpoint if it exists).
ckpt_path = MODELS_DIR / "deepfm_calib.pt"
hist_path = MODELS_DIR / "history_calib.json"

model = DeepFM(feature_config=feature_config, dnn_layers=list(DEFAULT_HPARAMS.dnn_layers), dropout=DEFAULT_HPARAMS.dropout)
trainer = Trainer(
    model=model,
    feature_config=feature_config,
    config=TrainerConfig(lr=1e-3, batch_size=2048, epochs=20, patience=5),
    checkpoint_path=ckpt_path,
    history_path=hist_path,
)
history = trainer.fit(train_df, val_df)
display(Markdown(f"**Best val AUC:** {trainer.best_val_auc:.5f} at epoch {trainer.best_epoch}"))

epoch=1 train_loss=1.24864 val_loss=0.23431 val_auc=0.76991 val_logloss=0.23427 lr=0.001000
epoch=2 train_loss=0.22805 val_loss=0.18602 val_auc=0.80692 val_logloss=0.18594 lr=0.001000
epoch=3 train_loss=0.20565 val_loss=0.18336 val_auc=0.81324 val_logloss=0.18326 lr=0.001000
epoch=4 train_loss=0.20125 val_loss=0.18325 val_auc=0.81397 val_logloss=0.18317 lr=0.001000
epoch=5 train_loss=0.19884 val_loss=0.18304 val_auc=0.81501 val_logloss=0.18296 lr=0.001000
epoch=6 train_loss=0.19670 val_loss=0.18375 val_auc=0.81352 val_logloss=0.18368 lr=0.001000
epoch=7 train_loss=0.19506 val_loss=0.18380 val_auc=0.81317 val_logloss=0.18373 lr=0.000500
epoch=8 train_loss=0.19328 val_loss=0.18361 val_auc=0.81318 val_logloss=0.18356 lr=0.000500
epoch=9 train_loss=0.19245 val_loss=0.18368 val_auc=0.81219 val_logloss=0.18363 lr=0.000250
epoch=10 train_loss=0.19118 val_loss=0.18417 val_auc=0.81176 val_logloss=0.18412 lr=0.000250
Early stopping triggered at epoch 10.


**Best val AUC:** 0.81501 at epoch 5

## 1) Reliability Diagram: Before vs After Platt Scaling

In [4]:
# Get raw (uncalibrated) predictions on the test set.
test_ds = AdClickDataset(test_df, trainer.feature_config)
test_loader = DataLoader(test_ds, batch_size=2048, shuffle=False, collate_fn=_collate_batch)
_, y_test, raw_probs = trainer._epoch_pass(test_loader, train=False)

# Platt-scaled predictions.
cal_probs = trainer.platt_scaler.calibrate(raw_probs)

# Compute ECE before and after.
ece_before = compute_ece(y_test, raw_probs, n_bins=10)
ece_after = compute_ece(y_test, cal_probs, n_bins=10)

display(Markdown(f"**ECE before Platt scaling:** `{ece_before.ece:.5f}`"))
display(Markdown(f"**ECE after Platt scaling:** `{ece_after.ece:.5f}`"))

**ECE before Platt scaling:** `0.00777`

**ECE after Platt scaling:** `0.00293`

In [5]:
def plot_reliability(before: "BinReliability", after: "BinReliability"):
    fig = go.Figure()
    midpoints_b = 0.5 * (before.bin_edges[:-1] + before.bin_edges[1:])
    midpoints_a = 0.5 * (after.bin_edges[:-1] + after.bin_edges[1:])

    mask_b = ~np.isnan(before.bin_mean_true)
    mask_a = ~np.isnan(after.bin_mean_true)

    fig.add_trace(go.Scatter(
        x=before.bin_mean_pred[mask_b], y=before.bin_mean_true[mask_b],
        mode="lines+markers", name=f"Before (ECE={before.ece:.4f})",
    ))
    fig.add_trace(go.Scatter(
        x=after.bin_mean_pred[mask_a], y=after.bin_mean_true[mask_a],
        mode="lines+markers", name=f"After Platt (ECE={after.ece:.4f})",
    ))
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines", name="Perfect",
        line=dict(dash="dash", color="gray"),
    ))
    fig.update_layout(
        title="Reliability Diagram: Before vs After Platt Scaling",
        xaxis_title="Mean predicted probability",
        yaxis_title="Observed click rate",
    )
    return fig

plot_reliability(ece_before, ece_after).show()

## 2) ECE Improvement Summary

In [6]:
ece_table = pd.DataFrame([
    {"Metric": "ECE (before Platt)", "Value": round(ece_before.ece, 5)},
    {"Metric": "ECE (after Platt)", "Value": round(ece_after.ece, 5)},
    {"Metric": "ECE reduction (%)", "Value": round(100 * (ece_before.ece - ece_after.ece) / max(ece_before.ece, 1e-9), 1)},
])
display(ece_table)

if ece_after.ece < 0.03:
    display(Markdown("**PASS** - ECE after Platt scaling < 0.03"))
else:
    display(Markdown(f"**WARNING** - ECE after Platt scaling = {ece_after.ece:.4f} (target < 0.03)"))

,Metric,Value
0,ECE (before Platt),0.00777
1,ECE (after Platt),0.00293
2,ECE reduction (%),62.30000


**PASS** - ECE after Platt scaling < 0.03

## 3) Position Invariance: PositionDebiaser

Predict the same (user, restaurant) pairs at positions 1 through 10. Calibrated+debiased predictions should be constant.

In [7]:
# Take a small sample of test rows for the demo.
sample = test_df.head(500).copy()
positions = list(range(1, 11))

debiaser = PositionDebiaser(
    model=trainer.model,
    feature_config=trainer.feature_config,
    scaler=trainer.platt_scaler,
)

raw_preds_per_pos = []
debiased_preds_per_pos = []

trainer.model.eval()
for pos in positions:
    pos_df = sample.copy()
    pos_df["ad_position"] = float(pos)

    # Raw model prediction (position-sensitive).
    ds = AdClickDataset(pos_df, trainer.feature_config)
    loader = DataLoader(ds, batch_size=2048, shuffle=False, collate_fn=_collate_batch)
    _, _, raw_p = trainer._epoch_pass(loader, train=False)
    raw_preds_per_pos.append(raw_p.mean())

    # Debiased + calibrated (position-invariant).
    deb_p = debiaser.predict(pos_df, calibrate=True)
    debiased_preds_per_pos.append(deb_p.mean())

raw_preds_per_pos = np.array(raw_preds_per_pos)
debiased_preds_per_pos = np.array(debiased_preds_per_pos)

print("Raw mean predictions by position:     ", np.round(raw_preds_per_pos, 5))
print("Debiased mean predictions by position: ", np.round(debiased_preds_per_pos, 5))

Raw mean predictions by position:      [0.16896 0.07026 0.04045 0.02672 0.02171 0.02059 0.01908 0.01827 0.01749
 0.01441]
Debiased mean predictions by position:  [0.19483 0.19483 0.19483 0.19483 0.19483 0.19483 0.19483 0.19483 0.19483
 0.19483]


In [8]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=positions, y=raw_preds_per_pos,
    mode="lines+markers", name="Raw (position-sensitive)",
))
fig.add_trace(go.Scatter(
    x=positions, y=debiased_preds_per_pos,
    mode="lines+markers", name="Debiased + calibrated",
))
fig.update_layout(
    title="Mean Predicted CTR by Ad Position",
    xaxis_title="Ad Position",
    yaxis_title="Mean predicted P(click)",
    yaxis_range=[0, max(raw_preds_per_pos.max(), 0.15) * 1.2],
)
fig.show()

In [9]:
# Verify position invariance numerically.
max_deviation = np.ptp(debiased_preds_per_pos)
display(Markdown(f"**Max deviation across positions:** `{max_deviation:.8f}`"))
assert max_deviation < 1e-5, f"Debiased predictions vary by {max_deviation:.6f}"
display(Markdown("**PASS** - Debiased predictions are constant across all 10 positions."))

**Max deviation across positions:** `0.00000000`

**PASS** - Debiased predictions are constant across all 10 positions.

## Acceptance Checklist

In [10]:
checks = {
    "ECE after Platt < 0.03": ece_after.ece < 0.03,
    "Position debiased invariance (max dev < 1e-5)": max_deviation < 1e-5,
}
for name, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    print(f"[{mark}] {name}")

scaler_path = MODELS_DIR / "platt_scaler.pkl"
if scaler_path.exists():
    size_kb = scaler_path.stat().st_size / 1024
    checks["PlattScaler pickle < 1 MB"] = size_kb < 1024
    print(f"[PASS] PlattScaler pickle size: {size_kb:.1f} KB")
else:
    print("[INFO] platt_scaler.pkl not found at MODELS_DIR (may be at deepfm_calib parent)")

if all(checks.values()):
    display(Markdown("### All acceptance criteria met."))

[PASS] ECE after Platt < 0.03
[PASS] Position debiased invariance (max dev < 1e-5)
[PASS] PlattScaler pickle size: 0.1 KB


### All acceptance criteria met.